In [1]:
# Cell 1 — Install Dependencies
!pip install openai langchain langchain-openai chromadb sentence-transformers newspaper3k newsapi-python python-dotenv -q

print("✅ Dependencies installed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 64.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.1/211.1 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14

In [3]:
# Cell 2 — Setup API Key Safely
from google.colab import userdata
import os

# Store your API key in Colab Secrets (safer than hardcoding)
# Click the 🔑 key icon on the left sidebar in Colab
# Add secret name: OPENAI_API_KEY
# Add your API key as the value

try:
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("✅ API key loaded from Colab Secrets!")
except:
    # Fallback — paste directly (only for testing)
    os.environ["OPENAI_API_KEY"] = "paste-your-key-here"
    print("⚠️ API key set directly — move to Colab Secrets later!")

✅ API key loaded from Colab Secrets!


In [11]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
response = llm.invoke([HumanMessage(content="Say 'InfoTrace RAG system is ready!' in one line")])
print(response.content)

InfoTrace RAG system is ready!


In [12]:
!pip install lxml_html_clean -q
print("✅ Fixed!")

✅ Fixed!


In [13]:
import chromadb
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
chroma_client = chromadb.Client()

# Get or create — won't fail if already exists
collection = chroma_client.get_or_create_collection(name="infotrace_knowledge")

print("✅ ChromaDB collection ready!")
print(f"📚 Embedding model: all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ ChromaDB collection ready!
📚 Embedding model: all-MiniLM-L6-v2


In [14]:
from newspaper import Article
import time

def fetch_verified_facts(topic):
    sources = [
        f"https://www.bbc.com/search?q={topic.replace(' ', '+')}",
        f"https://www.reuters.com/search/news?blob={topic.replace(' ', '+')}",
    ]

    articles = []

    # Sample verified articles for testing
    # (Real scraping will be added in production)
    sample_facts = [
        {
            "source": "BBC News",
            "title": f"Latest updates on {topic}",
            "content": f"According to verified reports, {topic} has been covered extensively by international media with confirmed facts and figures from official sources.",
            "url": "https://bbc.com"
        },
        {
            "source": "Reuters",
            "title": f"Reuters coverage of {topic}",
            "content": f"Reuters correspondents have verified key facts about {topic} through official government and institutional sources.",
            "url": "https://reuters.com"
        }
    ]

    print(f"✅ Fetched {len(sample_facts)} verified sources for: '{topic}'")
    return sample_facts

# Test it
topic = "flood in Assam"
facts = fetch_verified_facts(topic)
for f in facts:
    print(f"\n📰 Source: {f['source']}")
    print(f"   Title: {f['title']}")

✅ Fetched 2 verified sources for: 'flood in Assam'

📰 Source: BBC News
   Title: Latest updates on flood in Assam

📰 Source: Reuters
   Title: Reuters coverage of flood in Assam


In [15]:
def store_in_vectordb(articles, topic, source_type):
    for i, article in enumerate(articles):
        embedding = embedding_model.encode(article["content"]).tolist()

        collection.add(
            embeddings=[embedding],
            documents=[article["content"]],
            metadatas=[{
                "source": article["source"],
                "title": article["title"],
                "url": article["url"],
                "topic": topic,
                "source_type": source_type
            }],
            ids=[f"{source_type}_{topic.replace(' ', '_')}_{i}"]
        )

    print(f"✅ Stored {len(articles)} {source_type} documents in ChromaDB!")

# Test it
store_in_vectordb(facts, topic, "verified_news")

# Check whats in the database
print(f"\n📊 Total documents in ChromaDB: {collection.count()}")

✅ Stored 2 verified_news documents in ChromaDB!

📊 Total documents in ChromaDB: 2


In [16]:
def simulate_platform_posts(topic):
    platform_posts = {
        "facebook": [
            {
                "content": f"BREAKING: Government is hiding the real truth about {topic}! Share before they delete this!!",
                "likes": 1500,
                "shares": 890
            },
            {
                "content": f"My cousin who lives there says the situation with {topic} is much worse than what media is showing us.",
                "likes": 432,
                "shares": 211
            },
            {
                "content": f"Official reports confirm {topic} has affected thousands. Relief efforts are underway by local authorities.",
                "likes": 234,
                "shares": 89
            }
        ],
        "instagram": [
            {
                "content": f"Pray for everyone affected 🙏 {topic} is heartbreaking. Share awareness! #pray #help",
                "likes": 4500,
                "shares": 1200
            },
            {
                "content": f"Real footage of {topic} that mainstream media won't show you. Stay informed! #truth #wakeup",
                "likes": 2300,
                "shares": 670
            },
            {
                "content": f"Relief organizations are on ground helping victims of {topic}. Here is how you can help.",
                "likes": 1800,
                "shares": 430
            }
        ]
    }

    print(f"✅ Simulated posts for: '{topic}'")
    for platform, posts in platform_posts.items():
        print(f"   {platform}: {len(posts)} posts")

    return platform_posts

# Test it
platform_posts = simulate_platform_posts(topic)

✅ Simulated posts for: 'flood in Assam'
   facebook: 3 posts
   instagram: 3 posts


In [18]:
def analyze_platform_posts(platform_posts, collection):
    results = {}

    for platform, posts in platform_posts.items():
        platform_scores = []

        for post in posts:
            # Convert post to embedding
            post_embedding = embedding_model.encode(post["content"]).tolist()

            # Query ChromaDB for similar verified facts
            query_results = collection.query(
                query_embeddings=[post_embedding],
                n_results=1
            )

            # Get similarity score (1 - distance)
            distance = query_results["distances"][0][0]
            similarity = round(1 - distance, 4)

            platform_scores.append({
                "post": post["content"],
                "similarity_to_facts": similarity,
                "likes": post["likes"],
                "shares": post["shares"]
            })

        # Average similarity score for platform
        avg_similarity = round(
            sum(p["similarity_to_facts"] for p in platform_scores) / len(platform_scores), 4
        )

        results[platform] = {
            "posts": platform_scores,
            "average_similarity": avg_similarity,
            "total_posts": len(posts)
        }

        print(f"\n📊 {platform.upper()} Analysis:")
        print(f"   Average similarity to verified facts: {avg_similarity}")
        for i, p in enumerate(platform_scores):
            print(f"   Post {i+1}: {p['similarity_to_facts']} similarity | {p['likes']} likes")

    return results

# Test it
analysis_results = analyze_platform_posts(platform_posts, collection)



📊 FACEBOOK Analysis:
   Average similarity to verified facts: 0.4451
   Post 1: 0.3944 similarity | 1500 likes
   Post 2: 0.4644 similarity | 432 likes
   Post 3: 0.4766 similarity | 234 likes

📊 INSTAGRAM Analysis:
   Average similarity to verified facts: 0.196
   Post 1: 0.0699 similarity | 4500 likes
   Post 2: 0.4791 similarity | 2300 likes
   Post 3: 0.0391 similarity | 1800 likes


In [19]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

def generate_infotrace_report(topic, analysis_results):

    summary = ""
    for platform, data in analysis_results.items():
        summary += f"\n{platform.upper()}:\n"
        summary += f"  Average similarity to verified facts: {data['average_similarity']}\n"
        for i, post in enumerate(data["posts"]):
            summary += f"  Post {i+1}: similarity={post['similarity_to_facts']} | likes={post['likes']} | shares={post['shares']}\n"
            summary += f"  Content: {post['post'][:100]}...\n"

    prompt = f"""
You are InfoTrace, an intelligent information analysis system.

Topic being analyzed: "{topic}"

Here is how this topic is being discussed across social media platforms,
with similarity scores showing how closely each post aligns with verified facts
(1.0 = perfectly aligned, 0.0 = completely diverged from facts):

{summary}

Please provide:
1. A clear comparison of how each platform is discussing this topic
2. Which platform is most aligned with facts and which is most distorted
3. What kind of misinformation patterns you notice
4. Which posts are most dangerous (high distortion + high engagement)
5. A brief InfoTrace score summary for each platform

Be specific, insightful and use the data provided.
"""

    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content

# Generate report
print("🔍 Generating InfoTrace Report...\n")
report = generate_infotrace_report(topic, analysis_results)
print(report)

🔍 Generating InfoTrace Report...

1. Facebook posts generally have a higher average similarity to verified facts compared to Instagram posts. Facebook posts tend to provide more information about the flood in Assam, including official reports and relief efforts. On the other hand, Instagram posts focus more on emotional responses, sharing awareness, and questioning mainstream media coverage.

2. Facebook is most aligned with facts with an average similarity score of 0.4451, while Instagram is the most distorted with an average similarity score of 0.196.

3. Misinformation patterns on Facebook include claims of the government hiding the truth and exaggerating the situation, while on Instagram, there is a focus on emotional appeals, sharing unverified footage, and questioning mainstream media credibility.

4. The most dangerous posts are Post 1 on Facebook (similarity=0.3944, likes=1500, shares=890) and Post 2 on Instagram (similarity=0.4791, likes=2300, shares=670) as they have high eng

In [20]:
from transformers import pipeline

sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    return_all_scores=True
)

print("✅ Sentiment analyzer loaded!")

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

✅ Sentiment analyzer loaded!


In [22]:
def analyze_sentiment_per_platform(platform_posts):
    sentiment_results = {}

    for platform, posts in platform_posts.items():
        platform_sentiments = []

        for post in posts:
            # Get sentiment scores
            raw = sentiment_analyzer(post["content"][:512])

            # Handle both list formats
            if isinstance(raw[0], list):
                scores = raw[0]
            else:
                scores = raw

            # Extract scores
            sentiment_dict = {s["label"]: round(s["score"], 4) for s in scores}

            platform_sentiments.append({
                "post": post["content"][:80] + "...",
                "sentiment": sentiment_dict,
                "dominant": max(sentiment_dict, key=sentiment_dict.get),
                "likes": post["likes"],
                "shares": post["shares"]
            })

        # Calculate platform level averages
        avg_positive = round(sum(p["sentiment"].get("positive", 0) for p in platform_sentiments) / len(platform_sentiments), 4)
        avg_negative = round(sum(p["sentiment"].get("negative", 0) for p in platform_sentiments) / len(platform_sentiments), 4)
        avg_neutral  = round(sum(p["sentiment"].get("neutral", 0)  for p in platform_sentiments) / len(platform_sentiments), 4)

        sentiment_results[platform] = {
            "posts": platform_sentiments,
            "average": {
                "positive": avg_positive,
                "negative": avg_negative,
                "neutral":  avg_neutral
            }
        }

        print(f"\n💬 {platform.upper()} Sentiment:")
        print(f"   Positive: {avg_positive} | Negative: {avg_negative} | Neutral: {avg_neutral}")
        for i, p in enumerate(platform_sentiments):
            print(f"   Post {i+1}: {p['dominant']} | likes={p['likes']}")

    return sentiment_results

# Test it
sentiment_results = analyze_sentiment_per_platform(platform_posts)


💬 FACEBOOK Sentiment:
   Positive: 0.0 | Negative: 0.7237 | Neutral: 0.0
   Post 1: negative | likes=1500
   Post 2: negative | likes=432
   Post 3: negative | likes=234

💬 INSTAGRAM Sentiment:
   Positive: 0.0 | Negative: 0.1775 | Neutral: 0.4838
   Post 1: negative | likes=4500
   Post 2: neutral | likes=2300
   Post 3: neutral | likes=1800


In [24]:
!pip install bertopic -q
print("✅ BERTopic installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 10.9 MB/s eta 0:00:00
✅ BERTopic installed!


In [25]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

def analyze_topics_per_platform(platform_posts):
    topic_results = {}

    for platform, posts in platform_posts.items():
        # Extract post contents
        docs = [post["content"] for post in posts]

        # Need minimum 3 docs for BERTopic
        if len(docs) < 3:
            print(f"⚠️ {platform}: Not enough posts for topic modeling")
            continue

        # Use simple keyword extraction for small datasets
        vectorizer = CountVectorizer(
            ngram_range=(1, 2),
            stop_words="english",
            max_features=10
        )

        try:
            X = vectorizer.fit_transform(docs)
            keywords = vectorizer.get_feature_names_out()

            topic_results[platform] = {
                "dominant_keywords": list(keywords),
                "total_posts": len(docs)
            }

            print(f"\n🔍 {platform.upper()} Top Keywords:")
            print(f"   {', '.join(keywords[:8])}")

        except Exception as e:
            print(f"⚠️ {platform} topic modeling failed: {e}")

    return topic_results

# Test it
topic_results = analyze_topics_per_platform(platform_posts)

/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.



🔍 FACEBOOK Top Keywords:
   affected, affected thousands, assam, assam affected, assam share, assam worse, breaking government, confirm

🔍 INSTAGRAM Top Keywords:
   affected, assam, assam help, assam mainstream, awareness, awareness pray, flood, flood assam


In [26]:

def infotrace_orchestrator(user_topic):
    print(f"🔍 InfoTrace Analysis Started")
    print(f"📌 Topic: '{user_topic}'")
    print("="*50)

    # Step 1 — Fetch verified facts
    print("\n📰 Step 1: Fetching verified facts...")
    facts = fetch_verified_facts(user_topic)
    store_in_vectordb(facts, user_topic, "verified_news")

    # Step 2 — Collect platform posts
    print("\n📱 Step 2: Collecting platform posts...")
    posts = simulate_platform_posts(user_topic)

    # Step 3 — Similarity analysis
    print("\n🔎 Step 3: Analyzing similarity to verified facts...")
    analysis = analyze_platform_posts(posts, collection)

    # Step 4 — Sentiment analysis
    print("\n💬 Step 4: Analyzing sentiment per platform...")
    sentiment = analyze_sentiment_per_platform(posts)

    # Step 5 — Topic modeling
    print("\n🧠 Step 5: Extracting dominant narratives...")
    topics = analyze_topics_per_platform(posts)

    # Step 6 — Generate final report
    print("\n📄 Step 6: Generating InfoTrace Report...")
    report = generate_infotrace_report(user_topic, analysis)

    # Compile full results
    full_results = {
        "topic": user_topic,
        "similarity": analysis,
        "sentiment": sentiment,
        "topics": topics,
        "report": report
    }

    print("\n" + "="*50)
    print("✅ InfoTrace Analysis Complete!")
    print("="*50)
    print(f"\n{report}")

    return full_results

# Test the full pipeline with one prompt!
results = infotrace_orchestrator("flood in Assam")

🔍 InfoTrace Analysis Started
📌 Topic: 'flood in Assam'

📰 Step 1: Fetching verified facts...
✅ Fetched 2 verified sources for: 'flood in Assam'
✅ Stored 2 verified_news documents in ChromaDB!

📱 Step 2: Collecting platform posts...
✅ Simulated posts for: 'flood in Assam'
   facebook: 3 posts
   instagram: 3 posts

🔎 Step 3: Analyzing similarity to verified facts...

📊 FACEBOOK Analysis:
   Average similarity to verified facts: 0.4451
   Post 1: 0.3944 similarity | 1500 likes
   Post 2: 0.4644 similarity | 432 likes
   Post 3: 0.4766 similarity | 234 likes

📊 INSTAGRAM Analysis:
   Average similarity to verified facts: 0.196
   Post 1: 0.0699 similarity | 4500 likes
   Post 2: 0.4791 similarity | 2300 likes
   Post 3: 0.0391 similarity | 1800 likes

💬 Step 4: Analyzing sentiment per platform...

💬 FACEBOOK Sentiment:
   Positive: 0.0 | Negative: 0.7237 | Neutral: 0.0
   Post 1: negative | likes=1500
   Post 2: negative | likes=432
   Post 3: negative | likes=234

💬 INSTAGRAM Sentiment:


In [30]:
import json

# Use the actual filename that was uploaded
filename = "dataset_instagram-hashtag-scraper_2026-03-08_18-50-27-863.json"

with open(filename, "r", encoding="utf-8") as f:
    instagram_data = json.load(f)

print(f"✅ Loaded {len(instagram_data)} real Instagram posts!")
print(f"\n=== Sample Post ===")
print(json.dumps(instagram_data[0], indent=2))

✅ Loaded 20 real Instagram posts!

=== Sample Post ===
{
  "caption": "A husband and wife are at home.\n\"Honey, do you even hear me when I talk to you?\"\n\"Of course I do.\"\n\"And what did I just say?\"\n\"Well\u2026 the same thing you always say.\"\n\"And what is that?\"\n\"\u2018Do you even hear me when I talk to you?\u2019\"\n#proxy #mobileproxy #webscraping #seo #digitalmarketing #automation #business #tech #cybersecurity #datacollection #api #ecommerce #smm #proxyserver #residentialproxy #vpn #datacenter #webdata #developer #mobileproxyspace",
  "ownerFullName": "Mobileproxy.space",
  "ownerUsername": "mobileproxyspace",
  "url": "https://www.instagram.com/p/DVoS1uKD0_0/",
  "commentsCount": 0,
  "firstComment": "",
  "likesCount": 0,
  "timestamp": "2026-03-08T16:08:21.000Z",
  "hashtags": [
    "proxy",
    "mobileproxy",
    "webscraping",
    "seo",
    "digitalmarketing",
    "automation",
    "business",
    "tech",
    "cybersecurity",
    "datacollection",
    "api",
  

In [32]:
from google.colab import files
import json

print("Please upload new instagram file")
uploaded2 = files.upload()

# Get the filename automatically
filename2 = list(uploaded2.keys())[0]

with open(filename2, "r", encoding="utf-8") as f:
    instagram_data2 = json.load(f)

print(f"✅ Loaded {len(instagram_data2)} posts!")
print(f"\n=== Sample Post ===")
print(json.dumps(instagram_data2[0], indent=2))

Please upload new instagram file


Saving dataset_instagram-hashtag-scraper_2026-03-08_19-35-58-364.json to dataset_instagram-hashtag-scraper_2026-03-08_19-35-58-364.json
✅ Loaded 20 posts!

=== Sample Post ===
{
  "caption": "\ud83d\uded1\u0985\u09f0\u09c1\u09a3\u09be\u099a\u09b2\u09a4 \u09e7\u09e9\u099f\u09be \u09ac\u09c3\u09b9\u09ce\u200d \u09a8\u09a6\u09c0\u09ac\u09be\u09a8\u09cd\u09a7 \u09a8\u09bf\u09f0\u09cd\u09ae\u09be\u09a3\u09a4 \u0985\u09a8\u09c1\u09ae\u09cb\u09a6\u09a8\n\ud83d\uded1\u09a8\u09a6\u09c0\u09ac\u09be\u09a8\u09cd\u09a7\u09c7 \u0985\u09b8\u09ae\u09b2\u09c8 \u0995\u09a2\u09bc\u09bf\u09df\u09be\u09ac \u0985\u09b6\u09a8\u09bf \u09b8\u0982\u0995\u09c7\u09a4\n\n#ArunachalPradesh #bigdam #assamflood2024",
  "ownerFullName": "newsline925",
  "ownerUsername": "newsline925",
  "url": "https://www.instagram.com/p/DBO-6ubBFBk/",
  "commentsCount": 0,
  "firstComment": "",
  "likesCount": 0,
  "timestamp": "2024-10-17T17:45:15.000Z",
  "hashtags": [
    "ArunachalPradesh",
    "bigdam",
    "assamflood2024"
  ]

In [33]:
def extract_platform_posts_from_apify(data, platform):
    posts = []

    for item in data:
        caption = item.get("caption", "")

        # Skip empty captions
        if not caption or len(caption) < 10:
            continue

        posts.append({
            "content": caption,
            "likes": item.get("likesCount", 0),
            "shares": item.get("commentsCount", 0),
            "url": item.get("url", ""),
            "timestamp": item.get("timestamp", ""),
            "username": item.get("ownerUsername", "")
        })

    print(f"✅ Extracted {len(posts)} valid posts from {platform}")
    return posts

# Extract real Instagram posts
real_instagram_posts = extract_platform_posts_from_apify(instagram_data2, "instagram")

# Show sample
print(f"\n=== Sample Extracted Post ===")
print(f"Content: {real_instagram_posts[0]['content'][:100]}...")
print(f"Likes: {real_instagram_posts[0]['likes']}")
print(f"Username: {real_instagram_posts[0]['username']}")

✅ Extracted 20 valid posts from instagram

=== Sample Extracted Post ===
Content: 🛑অৰুণাচলত ১৩টা বৃহৎ‍ নদীবান্ধ নিৰ্মাণত অনুমোদন
🛑নদীবান্ধে অসমলৈ কঢ়িয়াব অশনি সংকেত

#ArunachalPrades...
Likes: 0
Username: newsline925


In [34]:
# Run real Instagram data through InfoTrace pipeline
real_platform_posts = {
    "instagram": real_instagram_posts
}

print("🔍 Running Real Instagram Data Through InfoTrace Pipeline...")
print("="*50)

# Step 1 — Fetch verified facts
print("\n📰 Step 1: Fetching verified facts...")
real_facts = fetch_verified_facts("assam flood 2024")
store_in_vectordb(real_facts, "assam flood 2024", "verified_news")

# Step 2 — Similarity analysis
print("\n🔎 Step 2: Analyzing similarity to verified facts...")
real_analysis = analyze_platform_posts(real_platform_posts, collection)

# Step 3 — Sentiment analysis
print("\n💬 Step 3: Analyzing sentiment...")
real_sentiment = analyze_sentiment_per_platform(real_platform_posts)

# Step 4 — Topic modeling
print("\n🧠 Step 4: Extracting narratives...")
real_topics = analyze_topics_per_platform(real_platform_posts)

# Step 5 — Generate report
print("\n📄 Step 5: Generating InfoTrace Report...")
real_report = generate_infotrace_report("assam flood 2024", real_analysis)

print("\n" + "="*50)
print("✅ Real Data Analysis Complete!")
print("="*50)
print(f"\n{real_report}")

🔍 Running Real Instagram Data Through InfoTrace Pipeline...

📰 Step 1: Fetching verified facts...
✅ Fetched 2 verified sources for: 'assam flood 2024'
✅ Stored 2 verified_news documents in ChromaDB!

🔎 Step 2: Analyzing similarity to verified facts...

📊 INSTAGRAM Analysis:
   Average similarity to verified facts: -0.0941
   Post 1: -0.3168 similarity | 0 likes
   Post 2: -0.3435 similarity | 0 likes
   Post 3: -0.3219 similarity | 24 likes
   Post 4: 0.0199 similarity | 135 likes
   Post 5: -0.0848 similarity | 22 likes
   Post 6: 0.0505 similarity | 0 likes
   Post 7: 0.0505 similarity | 0 likes
   Post 8: -0.0542 similarity | 8 likes
   Post 9: -0.4381 similarity | 71 likes
   Post 10: 0.07 similarity | 0 likes
   Post 11: 0.07 similarity | 0 likes
   Post 12: 0.0191 similarity | 33 likes
   Post 13: -0.2724 similarity | 0 likes
   Post 14: -0.1057 similarity | 12 likes
   Post 15: -0.042 similarity | 8 likes
   Post 16: -0.0149 similarity | 6 likes
   Post 17: -0.1528 similarity | 

In [35]:
!pip install newsapi-python -q
print("✅ NewsAPI installed!")

✅ NewsAPI installed!


In [36]:
from sentence_transformers import SentenceTransformer

# Replace English only model with multilingual model
# This understands 50+ languages including Hindi and Assamese
multilingual_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

print("✅ Multilingual embedding model loaded!")
print("🌍 Supports 50+ languages including Hindi, Assamese, Bengali")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Multilingual embedding model loaded!
🌍 Supports 50+ languages including Hindi, Assamese, Bengali


In [37]:
from newsapi import NewsApiClient
import os

# Initialize NewsAPI
newsapi = NewsApiClient(api_key="894106742b9b4bb2a683406fc0e3d9f0")

def fetch_real_verified_facts(topic):
    try:
        # Fetch real news articles
        response = newsapi.get_everything(
            q=topic,
            language="en",
            sort_by="relevancy",
            page_size=5
        )

        articles = []
        for article in response["articles"]:
            if article["content"] and len(article["content"]) > 50:
                articles.append({
                    "source": article["source"]["name"],
                    "title": article["title"],
                    "content": article["content"],
                    "url": article["url"]
                })

        print(f"✅ Fetched {len(articles)} real verified articles for: '{topic}'")
        return articles

    except Exception as e:
        print(f"⚠️ NewsAPI error: {e}")
        return []

# Test it
real_facts = fetch_real_verified_facts("assam flood 2024")
for f in real_facts:
    print(f"\n📰 {f['source']}: {f['title'][:60]}...")

✅ Fetched 4 real verified articles for: 'assam flood 2024'

📰 Nature.com: Integrating geospatial intelligence and machine learning for...

📰 The Times of India: BSEB starts D.El.Ed. 2026 online applications; deadlines and...

📰 ComputerWeekly.com: India AI Impact Summit begins...

📰 Syllad.com: New Guwahati airport terminal takes off, capacity jumps to 1...


In [38]:
# Try more specific queries
queries = [
    "Assam flood disaster 2024",
    "Assam flood relief northeast India",
    "Brahmaputra river flood Assam"
]

all_facts = []
for query in queries:
    facts = fetch_real_verified_facts(query)
    all_facts.extend(facts)

# Remove duplicates by URL
seen_urls = set()
unique_facts = []
for f in all_facts:
    if f["url"] not in seen_urls:
        seen_urls.add(f["url"])
        unique_facts.append(f)

print(f"\n✅ Total unique verified articles: {len(unique_facts)}")
for f in unique_facts:
    print(f"📰 {f['source']}: {f['title'][:70]}...")

✅ Fetched 1 real verified articles for: 'Assam flood disaster 2024'
✅ Fetched 1 real verified articles for: 'Assam flood relief northeast India'
✅ Fetched 2 real verified articles for: 'Brahmaputra river flood Assam'

✅ Total unique verified articles: 3
📰 Nature.com: Integrating geospatial intelligence and machine learning for flood sus...
📰 The Times of India: Assam to push for four more highway emergency facilities, says Himanta...
📰 The Times of India: Asian Development Bank clears $182 million loan for Assam flood manage...


In [39]:
import chromadb
from sentence_transformers import SentenceTransformer

# Fresh ChromaDB with multilingual model
chroma_client2 = chromadb.Client()
collection2 = chroma_client2.get_or_create_collection(name="infotrace_real")

# Store real verified facts
for i, article in enumerate(unique_facts):
    embedding = multilingual_model.encode(article["content"]).tolist()

    collection2.add(
        embeddings=[embedding],
        documents=[article["content"]],
        metadatas=[{
            "source": article["source"],
            "title": article["title"],
            "url": article["url"],
            "type": "verified_news"
        }],
        ids=[f"verified_{i}"]
    )

print(f"✅ Stored {len(unique_facts)} real verified articles!")
print(f"📊 Total documents in vector DB: {collection2.count()}")

✅ Stored 3 real verified articles!
📊 Total documents in vector DB: 3


In [40]:
def analyze_platform_posts_v2(platform_posts, collection, embedding_model):
    results = {}

    for platform, posts in platform_posts.items():
        platform_scores = []

        for post in posts:
            # Use multilingual model for embedding
            post_embedding = embedding_model.encode(post["content"]).tolist()

            # Query ChromaDB
            query_results = collection.query(
                query_embeddings=[post_embedding],
                n_results=1
            )

            distance = query_results["distances"][0][0]
            similarity = round(1 - distance, 4)

            platform_scores.append({
                "post": post["content"][:80] + "...",
                "similarity_to_facts": similarity,
                "likes": post["likes"],
                "shares": post["shares"]
            })

        avg_similarity = round(
            sum(p["similarity_to_facts"] for p in platform_scores) / len(platform_scores), 4
        )

        results[platform] = {
            "posts": platform_scores,
            "average_similarity": avg_similarity,
            "total_posts": len(posts)
        }

        print(f"\n📊 {platform.upper()} Analysis (Multilingual):")
        print(f"   Average similarity to verified facts: {avg_similarity}")
        for i, p in enumerate(platform_scores):
            print(f"   Post {i+1}: {p['similarity_to_facts']} similarity | {p['likes']} likes")

    return results

# Run with real data + multilingual model
real_analysis_v2 = analyze_platform_posts_v2(
    real_platform_posts,
    collection2,
    multilingual_model
)


📊 INSTAGRAM Analysis (Multilingual):
   Average similarity to verified facts: -14.6108
   Post 1: -12.6169 similarity | 0 likes
   Post 2: -13.6941 similarity | 0 likes
   Post 3: -16.5127 similarity | 24 likes
   Post 4: -11.1709 similarity | 135 likes
   Post 5: -15.1644 similarity | 22 likes
   Post 6: -17.3093 similarity | 0 likes
   Post 7: -17.3093 similarity | 0 likes
   Post 8: -13.0156 similarity | 8 likes
   Post 9: -9.9793 similarity | 71 likes
   Post 10: -21.4662 similarity | 0 likes
   Post 11: -21.4662 similarity | 0 likes
   Post 12: -11.3543 similarity | 33 likes
   Post 13: -15.335 similarity | 0 likes
   Post 14: -15.2811 similarity | 12 likes
   Post 15: -12.9722 similarity | 8 likes
   Post 16: -12.738 similarity | 6 likes
   Post 17: -10.5036 similarity | 19 likes
   Post 18: -16.4307 similarity | 0 likes
   Post 19: -9.8486 similarity | 0 likes
   Post 20: -18.0469 similarity | 2 likes


In [41]:
# Fix — use cosine similarity metric
chroma_client3 = chromadb.Client()
collection3 = chroma_client3.get_or_create_collection(
    name="infotrace_cosine",
    metadata={"hnsw:space": "cosine"}
)

# Re-store verified facts with cosine metric
for i, article in enumerate(unique_facts):
    embedding = multilingual_model.encode(article["content"]).tolist()
    collection3.add(
        embeddings=[embedding],
        documents=[article["content"]],
        metadatas=[{
            "source": article["source"],
            "title": article["title"],
            "url": article["url"],
            "type": "verified_news"
        }],
        ids=[f"verified_{i}"]
    )

print(f"✅ Rebuilt vector DB with cosine similarity!")
print(f"📊 Total documents: {collection3.count()}")

# Rerun analysis with cosine collection
real_analysis_v3 = analyze_platform_posts_v2(
    real_platform_posts,
    collection3,
    multilingual_model
)

✅ Rebuilt vector DB with cosine similarity!
📊 Total documents: 3

📊 INSTAGRAM Analysis (Multilingual):
   Average similarity to verified facts: 0.3551
   Post 1: 0.3689 similarity | 0 likes
   Post 2: 0.2428 similarity | 0 likes
   Post 3: 0.1049 similarity | 24 likes
   Post 4: 0.353 similarity | 135 likes
   Post 5: 0.3329 similarity | 22 likes
   Post 6: 0.3521 similarity | 0 likes
   Post 7: 0.3521 similarity | 0 likes
   Post 8: 0.3909 similarity | 8 likes
   Post 9: 0.4677 similarity | 71 likes
   Post 10: 0.3638 similarity | 0 likes
   Post 11: 0.3638 similarity | 0 likes
   Post 12: 0.2921 similarity | 33 likes
   Post 13: 0.3048 similarity | 0 likes
   Post 14: 0.3369 similarity | 12 likes
   Post 15: 0.4017 similarity | 8 likes
   Post 16: 0.4051 similarity | 6 likes
   Post 17: 0.4367 similarity | 19 likes
   Post 18: 0.3251 similarity | 0 likes
   Post 19: 0.4864 similarity | 0 likes
   Post 20: 0.421 similarity | 2 likes


In [42]:
# Generate final report with real data
print("📄 Generating Real Data InfoTrace Report...")
print("="*50)

real_report_v2 = generate_infotrace_report(
    "Assam flood 2024",
    real_analysis_v3
)

print(real_report_v2)

📄 Generating Real Data InfoTrace Report...
1. Instagram posts vary in their discussion of the "Assam flood 2024" topic. Some posts focus on providing updates and information related to the floods, while others seem to be more general or unrelated to the topic.

2. The post with the highest similarity score to verified facts is Post 19 with a similarity score of 0.4864, discussing flood updates in Dhubri. The post with the lowest similarity score is Post 3 with a similarity score of 0.1049, which seems to be unrelated to the topic of Assam floods.

3. Misinformation patterns observed include posts that are not directly related to the topic of Assam floods, posts that focus on personal content or unrelated topics, and posts that may not provide accurate information about the situation on the ground.

4. The most dangerous posts in terms of misinformation and engagement are Post 9 with a similarity score of 0.4677 and 71 likes, and Post 19 with a similarity score of 0.4864. These posts ha

In [43]:
from google.colab import files
import json

print("Please upload Facebook posts JSON file")
uploaded3 = files.upload()

filename3 = list(uploaded3.keys())[0]

with open(filename3, "r", encoding="utf-8") as f:
    facebook_data = json.load(f)

print(f"✅ Loaded {len(facebook_data)} real Facebook posts!")
print(f"\n=== Sample Post ===")
print(json.dumps(facebook_data[0], indent=2))

Please upload Facebook posts JSON file


Saving dataset_facebook-posts-scraper_2026-03-08_20-11-17-089.json to dataset_facebook-posts-scraper_2026-03-08_20-11-17-089.json
✅ Loaded 20 real Facebook posts!

=== Sample Post ===
{
  "media": [
    {
      "thumbnail": "https://scontent.fjbr1-1.fna.fbcdn.net/v/t39.30808-6/647154741_1376003821238644_7571354705712265008_n.jpg?stp=dst-jpg_s640x640_tt6&_nc_cat=101&ccb=1-7&_nc_sid=13d280&_nc_ohc=XkX2Dg2chNkQ7kNvwEiGviX&_nc_oc=AdnnEaAhtAhCmCO0zg2zvSIOe0ER_QQnmDrJO6uCUeKpvRBUAn-DKzG5vZEuDzfHS60&_nc_zt=23&_nc_ht=scontent.fjbr1-1.fna&_nc_gid=YV_LZMJ_YgMo0MlDRFlPhQ&_nc_ss=8&oh=00_AfzFhZTZ3BqGx83oCtXmJvL8pqa8Ce6U7jnEXjfhkUb0vA&oe=69B3BA19",
      "__typename": "Photo",
      "__isMedia": "Photo",
      "accent_color": "FF000000",
      "photo_product_tags": [],
      "photo_image": {
        "uri": "https://scontent.fjbr1-1.fna.fbcdn.net/v/t39.30808-6/647154741_1376003821238644_7571354705712265008_n.jpg?stp=dst-jpg_s640x640_tt6&_nc_cat=101&ccb=1-7&_nc_sid=13d280&_nc_ohc=XkX2Dg2chNkQ7kNvwEiGv

In [44]:
def filter_relevant_posts(data, keywords):
    relevant = []

    for post in data:
        text = post.get("text", "")
        if not text:
            continue

        # Check if any keyword exists in post text
        text_lower = text.lower()
        if any(keyword.lower() in text_lower for keyword in keywords):
            relevant.append({
                "content": text,
                "likes": post.get("likes", 0),
                "shares": post.get("shares", 0),
                "url": post.get("url", ""),
            })

    return relevant

# Filter flood related posts
flood_keywords = [
    "flood", "floods", "flooding", "river",
    "brahmaputra", "disaster", "relief",
    "assam flood", "inundated", "embankment"
]

facebook_flood_posts = filter_relevant_posts(facebook_data, flood_keywords)

print(f"✅ Total posts: {len(facebook_data)}")
print(f"🌊 Flood related posts: {len(facebook_flood_posts)}")

if facebook_flood_posts:
    print(f"\n=== Sample Relevant Post ===")
    print(facebook_flood_posts[0]["content"][:200])

✅ Total posts: 20
🌊 Flood related posts: 3

=== Sample Relevant Post ===
Aiming to extend financial relief to lakhs of families, Assam Chief Minister Himanta Biswa Sarma, on Sunday, announced that financial assistance of Rs 9,000 will be provided to around 40 lakh benefici


In [45]:
# Run real cross platform analysis
real_platform_posts_v2 = {
    "facebook": facebook_flood_posts,
    "instagram": real_instagram_posts
}

print("🔍 Running Real Cross-Platform InfoTrace Analysis...")
print("="*50)

# Similarity analysis
print("\n🔎 Analyzing similarity to verified facts...")
real_analysis_final = analyze_platform_posts_v2(
    real_platform_posts_v2,
    collection3,
    multilingual_model
)

# Sentiment analysis
print("\n💬 Analyzing sentiment per platform...")
real_sentiment_final = analyze_sentiment_per_platform(real_platform_posts_v2)

# Topic modeling
print("\n🧠 Extracting narratives per platform...")
real_topics_final = analyze_topics_per_platform(real_platform_posts_v2)

# Generate final report
print("\n📄 Generating Final InfoTrace Report...")
final_report = generate_infotrace_report(
    "Assam flood 2024",
    real_analysis_final
)

print("\n" + "="*50)
print("✅ Real Cross-Platform Analysis Complete!")
print("="*50)
print(f"\n{final_report}")

🔍 Running Real Cross-Platform InfoTrace Analysis...

🔎 Analyzing similarity to verified facts...

📊 FACEBOOK Analysis (Multilingual):
   Average similarity to verified facts: 0.2204
   Post 1: 0.26 similarity | 121 likes
   Post 2: 0.2455 similarity | 3 likes
   Post 3: 0.1557 similarity | 31 likes

📊 INSTAGRAM Analysis (Multilingual):
   Average similarity to verified facts: 0.3551
   Post 1: 0.3689 similarity | 0 likes
   Post 2: 0.2428 similarity | 0 likes
   Post 3: 0.1049 similarity | 24 likes
   Post 4: 0.353 similarity | 135 likes
   Post 5: 0.3329 similarity | 22 likes
   Post 6: 0.3521 similarity | 0 likes
   Post 7: 0.3521 similarity | 0 likes
   Post 8: 0.3909 similarity | 8 likes
   Post 9: 0.4677 similarity | 71 likes
   Post 10: 0.3638 similarity | 0 likes
   Post 11: 0.3638 similarity | 0 likes
   Post 12: 0.2921 similarity | 33 likes
   Post 13: 0.3048 similarity | 0 likes
   Post 14: 0.3369 similarity | 12 likes
   Post 15: 0.4017 similarity | 8 likes
   Post 16: 0.405